# Phase 1 optimizer -- backend cross-validation (SLSQP vs IPOPT vs ForwardSimulator)

This notebook is one of 7 split from `phase1_optimizer.ipynb` (kept alongside these as a single-file reference/narrative version).
Equation/section citations here follow `docs/tt pacing optimizer.md`, the
project's canonical numbering, written as "(design-doc Eq. N)" /
"(design-doc Section N.M)"; display-unit conventions (km/min/km-h⁻¹) are
implemented in `phase1_common.py`'s `as_km`/`as_min`/`as_kmh` helpers, shared
by all 7 notebooks. This one covers
**cross-backend and cross-simulator correctness checks, independent of the solver's own success flag**.

It is fully self-contained: every rider/course/baseline it needs is
rebuilt here via `phase1_common.py` (shared by all 7 split notebooks),
so it runs standalone from a fresh kernel without any other notebook
having run first. Courses used: flat, rolling (synthetic) + Giro10, TdF16, TARA (real). Approx. runtime: ~6-9 min.

In [1]:
import sys
sys.path.insert(0, ".")
import phase1_common as pc

pc.print_rider_summary()

reference_rider        mass= 72.0 kg  CP= 280.0 W  W'= 20.0 kJ  CdA=0.25 m^2  P_max=   900 W
evenepoel_like_rider   mass= 63.5 kg  CP= 425.0 W  W'= 20.0 kJ  CdA=0.21 m^2  P_max=  1400 W
ganna_like_rider       mass= 82.0 kg  CP= 480.0 W  W'= 20.0 kJ  CdA=0.19 m^2  P_max=  1600 W


In [2]:
courses = pc.load_real_courses()
giro10_course, tdf16_course, tara_course = courses["giro10"], courses["tdf16"], courses["tara"]
pc.print_real_course_summary(courses)

Giro 2026 Stage 10   n_nodes= 800  smoothing_length_m=  150 m  mean|grade|= 1.04%  max|grade|=13.20%
TdF 2026 Stage 16    n_nodes= 800  smoothing_length_m=  400 m  mean|grade|= 5.30%  max|grade|=18.53%
TARA 2026 Stage 3    n_nodes= 800  smoothing_length_m=  250 m  mean|grade|= 3.05%  max|grade|=17.56%


## Backend cross-validation

The commit's actual correctness argument is not "the solver reported
success" -- at the mesh resolution this project adopted, that flag stops
reliably indicating a real problem. Instead, two independent checks are
used:

1. **Cross-backend agreement**: SLSQP and IPOPT are different NLP
   algorithms (SLSQP: sequential quadratic programming; IPOPT: interior
   point) consuming the *same* analytic objective/constraint/Jacobian
   from `CollocationProblem`. If they agree closely on the optimal time,
   that is strong evidence the transcription and gradients are correct,
   independent of either solver's internal convergence bookkeeping.
2. **Independent re-simulation**: the NLP's own power plan, replayed
   through `ForwardSimulator` (a completely separate integrator, RK4 in
   the distance domain), should reproduce essentially the same total
   time.

Both are demonstrated below on two synthetic courses (a near-flat one and
a gently rolling one) and three real GPX courses (4.3-4.5), each course's
own Hermite-Simpson/SLSQP baseline rebuilt here as the SLSQP side of the
comparison.

### 4.1 Flat course

In [3]:
flat_course = pc.build_flat_course()
opt_hs_flat, res_hs_flat = pc.build_hs_baseline(pc.reference_rider, flat_course, pc.calm_wind, pc.N_INTERVALS_BASIC)

rel_diff_backend_flat, rel_diff_sim_flat = pc.run_backend_and_sim_crosscheck(
    pc.reference_rider, flat_course, pc.calm_wind, res_hs_flat, pc.N_INTERVALS_BASIC, "flat"
)


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************



[flat] SLSQP: T = 57.1178 min  (success=False)
[flat] IPOPT: T = 57.1193 min  (success=False)
[flat] Relative difference: 0.000027  (test_ipopt_agrees_with_slsqp_within_tolerance gate: < 1e-3)
[flat] NLP (SLSQP) time:      57.1178 min
[flat] ForwardSimulator time: 57.1671 min
[flat] Relative difference: 0.000863  (test_slsqp_cross_validates_against_forward_simulator gate: < 0.005)


### 4.2 Rolling course

In [4]:
rolling_course = pc.build_rolling_course()
opt_hs_rolling, res_hs_rolling = pc.build_hs_baseline(pc.reference_rider, rolling_course, pc.calm_wind, pc.N_INTERVALS_BASIC)

rel_diff_backend_rolling, rel_diff_sim_rolling = pc.run_backend_and_sim_crosscheck(
    pc.reference_rider, rolling_course, pc.calm_wind, res_hs_rolling, pc.N_INTERVALS_BASIC, "rolling"
)

[rolling] SLSQP: T = 34.0558 min  (success=False)
[rolling] IPOPT: T = 34.0450 min  (success=False)
[rolling] Relative difference: 0.000317  (test_ipopt_agrees_with_slsqp_within_tolerance gate: < 1e-3)
[rolling] NLP (SLSQP) time:      34.0558 min
[rolling] ForwardSimulator time: 34.1290 min
[rolling] Relative difference: 0.002150  (test_slsqp_cross_validates_against_forward_simulator gate: < 0.005)


Both checks pass comfortably inside their test gates on both courses, even
though `success` may individually be `False` above (SLSQP frequently
reports `False` at this mesh resolution while still landing on a solution
both cross-checks validate). This is exactly the design decision the
commit documents: gate correctness on cross-backend agreement and
independent re-simulation, not on the solver's own flag.

### 4.3 Giro 2026 Stage 10 (real, flat)

In [5]:
opt_hs_giro10, res_hs_giro10 = pc.build_hs_baseline(pc.ganna_like_rider, giro10_course, pc.calm_wind, pc.N_INTERVALS_GIRO10)

rel_diff_backend_giro10, rel_diff_sim_giro10 = pc.run_backend_and_sim_crosscheck(
    pc.ganna_like_rider, giro10_course, pc.calm_wind, res_hs_giro10, pc.N_INTERVALS_GIRO10, "giro10"
)

[giro10] SLSQP: T = 42.5510 min  (success=False)
[giro10] IPOPT: T = 42.9120 min  (success=False)
[giro10] Relative difference: 0.008486  (test_ipopt_agrees_with_slsqp_within_tolerance gate: < 1e-3)
[giro10] NLP (SLSQP) time:      42.5510 min
[giro10] ForwardSimulator time: 42.5155 min
[giro10] Relative difference: 0.000835  (test_slsqp_cross_validates_against_forward_simulator gate: < 0.005)


### 4.4 TdF 2026 Stage 16 (real, mountainous)

The steep-grade, hairpin-heavy course is the sternest test in this notebook for whether SLSQP and IPOPT still agree away from gentle terrain.

In [6]:
opt_hs_tdf16, res_hs_tdf16 = pc.build_hs_baseline(pc.evenepoel_like_rider, tdf16_course, pc.calm_wind, pc.N_INTERVALS_TDF16)

rel_diff_backend_tdf16, rel_diff_sim_tdf16 = pc.run_backend_and_sim_crosscheck(
    pc.evenepoel_like_rider, tdf16_course, pc.calm_wind, res_hs_tdf16, pc.N_INTERVALS_TDF16, "tdf16"
)

[tdf16] SLSQP: T = 37.1917 min  (success=False)
[tdf16] IPOPT: T = 37.1476 min  (success=False)
[tdf16] Relative difference: 0.001184  (test_ipopt_agrees_with_slsqp_within_tolerance gate: < 1e-3)
[tdf16] NLP (SLSQP) time:      37.1917 min
[tdf16] ForwardSimulator time: 37.2260 min
[tdf16] Relative difference: 0.000924  (test_slsqp_cross_validates_against_forward_simulator gate: < 0.005)


### 4.5 TARA 2026 Stage 3 (real, rolling/TTT)

In [7]:
opt_hs_tara, res_hs_tara = pc.build_hs_baseline(pc.reference_rider, tara_course, pc.calm_wind, pc.N_INTERVALS_TARA)

rel_diff_backend_tara, rel_diff_sim_tara = pc.run_backend_and_sim_crosscheck(
    pc.reference_rider, tara_course, pc.calm_wind, res_hs_tara, pc.N_INTERVALS_TARA, "tara"
)

[tara] SLSQP: T = 45.4841 min  (success=False)
[tara] IPOPT: T = 42.7715 min  (success=False)
[tara] Relative difference: 0.059639  (test_ipopt_agrees_with_slsqp_within_tolerance gate: < 1e-3)
[tara] NLP (SLSQP) time:      45.4841 min
[tara] ForwardSimulator time: 44.2645 min
[tara] Relative difference: 0.026813  (test_slsqp_cross_validates_against_forward_simulator gate: < 0.005)


Real terrain complicates this pair of gates in a way neither synthetic
course does. The independent-re-simulation check (NLP vs
ForwardSimulator) still holds up on two of the three: Giro10 (0.08%) and
TdF16 (0.09%) both stay comfortably under the 0.5% test gate, same as the
synthetic courses. TARA does not -- its own SLSQP plan disagrees with its
own re-simulation by 2.68%, more than 5x the gate. The cross-backend check
(SLSQP vs IPOPT) is looser still: none of the three real courses reach the
test suite's tight <0.1% bound (which, per `tests/test_phase_1.py`, is
only ever asserted against the synthetic flat-course fixture) -- Giro10
disagrees by 0.85%, TdF16 by 0.12%, TARA by 5.96%. TARA's disagreement is
large enough, and consistent enough with the launch-mesh fragility seen
again in the constrained-smoothing notebook's TARA subsection, that SLSQP
is plausibly landing on a materially different local solution on this
course than IPOPT does -- not simply a looser numerical match. The lesson
these tolerances teach (this section's opening) generalizes; the specific
numeric bar the test suite enforces does not automatically transfer to
courses the test suite was never written against. (TARA's simulator
disagreement is mildly sensitive to `smoothing_length_m` -- 3.58% at
150 m down to 2.24% at 300 m -- but never reaches the 0.5% gate at any
value tried, so this is a property of the course/rider/mesh combination,
not a fixable smoothing choice.)